# Ollama Embedding Benchmark

Compare the embedding sweep while holding the generation model fixed.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "main.py").exists():
    for parent in Path.cwd().resolve().parents:
        if (parent / "main.py").exists():
            repo_root = parent
            break

sys.path.insert(0, str(repo_root))

from eval.embedding_benchmark import EmbeddingBenchmarkConfig, run_embedding_benchmark
from helpers.experiment_models import DEFAULT_GENERATION_MODEL, EMBEDDING_MODEL_SWEEP

os.environ.setdefault("OLLAMA_ENDPOINT", "http://10.0.0.201:8000")
os.environ.setdefault("INPUT_BASE_DIR", str(repo_root / "data" / "evidence" / "mimic_discharge_subset"))

generation_model = os.environ.get("BENCHMARK_GENERATION_MODEL", DEFAULT_GENERATION_MODEL)
embedding_model_sweep = list(EMBEDDING_MODEL_SWEEP)
sample_size = 5
use_umls = os.environ.get("UMLS_ENABLED", "true").strip().lower() == "true"
schema_guided = os.environ.get("INDEX_SCHEMA_GUIDED", "false").strip().lower() == "true"
mimic_csv = repo_root / "data" / "mimic_iv_note" / "discharge.csv"

print("OLLAMA_ENDPOINT:", os.environ["OLLAMA_ENDPOINT"])
print("generation_model:", generation_model)
print("embedding_model_sweep:", embedding_model_sweep)


In [ ]:
results = await run_embedding_benchmark(
    EmbeddingBenchmarkConfig(
        input_dir=Path(os.environ["INPUT_BASE_DIR"]),
        output_root=repo_root / "output" / "ollama_embedding_benchmark",
        generation_model=generation_model,
        embedding_models=tuple(embedding_model_sweep),
        use_umls=use_umls,
        schema_guided=schema_guided,
        mimic_csv=(mimic_csv if mimic_csv.exists() else None),
        sample_size=sample_size,
    )
)

pd.DataFrame(result.__dict__ for result in results).sort_values(["exact_match", "mean_query_seconds"], ascending=[False, True])
